# Tokenizadores

In [887]:
from tokenizers import Tokenizer
import torch

In [888]:
tokenizador = Tokenizer.from_file("./data/tokenizer.json")
texto = "O gato dorme"

tokens = [5, 8, 2, 9, 4, 7]
context_size = 3
stride = 1

In [889]:
resultado = tokenizador.encode(texto)
# ['O', 'Ġga', 'to', 'Ġdor', 'me'] => Ġ representa espaço
print(resultado.tokens)
print(resultado.ids)

['O', 'Ġga', 'to', 'Ġdor', 'me']
[47, 4326, 287, 4956, 547]


In [890]:
texto_reconstruido = tokenizador.decode(resultado.ids)
print(texto_reconstruido)
print(texto_reconstruido == texto)

O gato dorme
True


# Ids

In [891]:
entrada = resultado.ids[:-1]
alvo = resultado.ids[1:]

print(entrada, alvo)

[47, 4326, 287, 4956] [4326, 287, 4956, 547]


# Stride

In [892]:
for i in range(0, len(tokens) - context_size, stride):
    print("i = ", i)
    x = tokens[i: i + context_size]
    y = tokens[i + 1: i + context_size + 1]
    print(x, y)

i =  0
[5, 8, 2] [8, 2, 9]
i =  1
[8, 2, 9] [2, 9, 4]
i =  2
[2, 9, 4] [9, 4, 7]


# Tensores

In [893]:
x = []
y = []

for i in range(0, len(tokens) - context_size, stride):
    print("i = ", i)
    x.append(tokens[i: i + context_size])
    y.append(tokens[i + 1: i + context_size + 1])
    print(x, y)
    
tensor_x = torch.tensor(x, dtype=torch.long)
tensor_y = torch.tensor(y, dtype=torch.long)
print(tensor_x, tensor_y)
print(tensor_x.shape)
print(tensor_y.shape)

i =  0
[[5, 8, 2]] [[8, 2, 9]]
i =  1
[[5, 8, 2], [8, 2, 9]] [[8, 2, 9], [2, 9, 4]]
i =  2
[[5, 8, 2], [8, 2, 9], [2, 9, 4]] [[8, 2, 9], [2, 9, 4], [9, 4, 7]]
tensor([[5, 8, 2],
        [8, 2, 9],
        [2, 9, 4]]) tensor([[8, 2, 9],
        [2, 9, 4],
        [9, 4, 7]])
torch.Size([3, 3])
torch.Size([3, 3])


# Batch

In [894]:
batch_size = 2


def batching(tensor_list, size=batch_size):
    batch = []
    for i in range(0, len(tensor_list), size):
        batch.append(tensor_list[i : i + size])

    return batch


batched_x = batching(tensor_x)
batched_y = batching(tensor_y)

print(batched_x, batched_y)
print(len(batched_x), len(batched_y))

[tensor([[5, 8, 2],
        [8, 2, 9]]), tensor([[2, 9, 4]])] [tensor([[8, 2, 9],
        [2, 9, 4]]), tensor([[9, 4, 7]])]
2 2


# Embeddings

In [895]:
num_embeddings = 10
num_dimensions = 4

embeddings = torch.nn.Embedding(num_embeddings, num_dimensions)

In [896]:
token_vectors = embeddings(batched_x[0])

print(token_vectors)
print("Tabela:", embeddings.weight.shape)
print("Entrada:", batched_x[0].shape)
print("Resultado:", token_vectors.shape)
print(embeddings.weight[8])

tensor([[[-3.2523e-04, -1.3805e+00,  6.9527e-01,  8.8186e-01],
         [ 6.9996e-01, -2.1218e+00,  9.7620e-01, -6.9381e-01],
         [ 1.4610e+00, -2.0504e-02, -1.8074e+00,  1.0274e+00]],

        [[ 6.9996e-01, -2.1218e+00,  9.7620e-01, -6.9381e-01],
         [ 1.4610e+00, -2.0504e-02, -1.8074e+00,  1.0274e+00],
         [-1.8295e+00,  4.6848e-01, -4.4149e-01, -1.0910e+00]]],
       grad_fn=<EmbeddingBackward0>)
Tabela: torch.Size([10, 4])
Entrada: torch.Size([2, 3])
Resultado: torch.Size([2, 3, 4])
tensor([ 0.7000, -2.1218,  0.9762, -0.6938], grad_fn=<SelectBackward0>)


# Posição

In [897]:
num_linhas = 3
num_componentes = 4

posicao = torch.nn.Embedding(num_linhas, num_componentes)
value = torch.arange(num_linhas)

In [898]:
position_vectors = posicao(value)
print(position_vectors, position_vectors.shape)

tensor([[ 0.9049,  2.0174, -2.3820,  0.4478],
        [ 0.2690,  0.1906, -0.2418,  2.7115],
        [-0.6033,  0.7381,  1.5657,  2.4367]], grad_fn=<EmbeddingBackward0>) torch.Size([3, 4])


In [899]:
input_vectors = position_vectors + token_vectors
print(input_vectors, input_vectors.shape)
print("Tokens:", token_vectors)
print("Posições:", position_vectors)

tensor([[[ 0.9046,  0.6368, -1.6867,  1.3296],
         [ 0.9690, -1.9312,  0.7344,  2.0177],
         [ 0.8577,  0.7176, -0.2418,  3.4642]],

        [[ 1.6049, -0.1044, -1.4058, -0.2461],
         [ 1.7300,  0.1701, -2.0493,  3.7389],
         [-2.4328,  1.2066,  1.1242,  1.3458]]], grad_fn=<AddBackward0>) torch.Size([2, 3, 4])
Tokens: tensor([[[-3.2523e-04, -1.3805e+00,  6.9527e-01,  8.8186e-01],
         [ 6.9996e-01, -2.1218e+00,  9.7620e-01, -6.9381e-01],
         [ 1.4610e+00, -2.0504e-02, -1.8074e+00,  1.0274e+00]],

        [[ 6.9996e-01, -2.1218e+00,  9.7620e-01, -6.9381e-01],
         [ 1.4610e+00, -2.0504e-02, -1.8074e+00,  1.0274e+00],
         [-1.8295e+00,  4.6848e-01, -4.4149e-01, -1.0910e+00]]],
       grad_fn=<EmbeddingBackward0>)
Posições: tensor([[ 0.9049,  2.0174, -2.3820,  0.4478],
        [ 0.2690,  0.1906, -0.2418,  2.7115],
        [-0.6033,  0.7381,  1.5657,  2.4367]], grad_fn=<EmbeddingBackward0>)


# Atenção

In [900]:
camadas_lineares = [
    torch.nn.Linear(4, 4, bias=False),
    torch.nn.Linear(4, 4, bias=False),
    torch.nn.Linear(4, 4, bias=False),
]

chamadas = []

for linear in camadas_lineares:
    chamadas.append(linear(input_vectors))

queries, keys, values = chamadas
chamadas = []

print(queries, keys, values)

tensor([[[-1.5018e+00, -2.8471e-02,  3.1008e-01, -1.6610e+00],
         [-9.4486e-01,  1.1694e+00, -1.8604e-01,  1.1737e-03],
         [-1.7715e+00,  7.4340e-01,  6.8663e-01, -2.1631e+00]],

        [[-9.4488e-01, -6.2849e-01, -2.5697e-01, -5.3927e-01],
         [-2.9486e+00,  7.9672e-01,  5.6156e-01, -2.8697e+00],
         [ 6.5193e-01,  9.4569e-01,  8.2703e-01, -4.8496e-01]]],
       grad_fn=<UnsafeViewBackward0>) tensor([[[-1.0408, -1.2373, -0.6870,  0.6660],
         [ 0.3243, -0.6307, -1.1926,  0.2657],
         [-1.2031, -1.1114, -0.8462,  0.7694]],

        [[-0.4463, -1.1661, -0.7459,  0.0286],
         [-1.5874, -2.2180, -1.6743,  1.3083],
         [-0.1632,  1.2575,  1.0563,  0.4712]]], grad_fn=<UnsafeViewBackward0>) tensor([[[-1.0245,  0.1130, -1.1906, -0.1174],
         [ 1.3978,  0.4402, -0.6703, -1.2132],
         [-0.3243, -0.9743, -1.8498, -1.5143]],

        [[-0.5524,  1.1460, -0.6406,  0.1957],
         [-0.8544,  0.0067, -2.3923, -1.0853],
         [-0.1186, -2.2436

In [901]:
resultado_produto_escalar = []
for i in range(3):
    resultado_produto_escalar.append(sum(queries[0, 1] * keys[0, i]))
    
tensor_pe = torch.stack(resultado_produto_escalar)
half_tensor_pe = tensor_pe/2
print(resultado_produto_escalar)
print(tensor_pe)
print(half_tensor_pe)


[tensor(-0.3349, grad_fn=<AddBackward0>), tensor(-0.8217, grad_fn=<AddBackward0>), tensor(-0.0045, grad_fn=<AddBackward0>)]
tensor([-0.3349, -0.8217, -0.0045], grad_fn=<StackBackward0>)
tensor([-0.1674, -0.4109, -0.0022], grad_fn=<DivBackward0>)


In [902]:
mascara = half_tensor_pe.clone()
mascara[2] = float("-inf")
print(mascara)

tensor([-0.1674, -0.4109,    -inf], grad_fn=<CopySlices>)


In [903]:
attention_weights = torch.softmax(mascara, dim=0)
print(attention_weights)
print(attention_weights.sum())

tensor([0.5606, 0.4394, 0.0000], grad_fn=<SoftmaxBackward0>)
tensor(1., grad_fn=<SumBackward0>)


In [904]:
weighted_vectors = []
for i in range(3):
    weighted_vectors.append(attention_weights[i]*values[0, i])
    
weighted_vectors = torch.stack(weighted_vectors)
print(weighted_vectors)
print(weighted_vectors.sum(dim=0))

tensor([[-0.5743,  0.0633, -0.6674, -0.0658],
        [ 0.6143,  0.1934, -0.2946, -0.5331],
        [-0.0000, -0.0000, -0.0000, -0.0000]], grad_fn=<StackBackward0>)
tensor([ 0.0400,  0.2568, -0.9620, -0.5989], grad_fn=<SumBackward1>)


# Implementando atenção

In [905]:
chamadas = []
camadas_lineares = [
    torch.nn.Linear(4, 4, bias=False),
    torch.nn.Linear(4, 4, bias=False),
    torch.nn.Linear(4, 4, bias=False),
]

for linear in camadas_lineares:
    chamadas.append(linear(input_vectors))

queries, keys, values = chamadas

In [906]:
# deprecated
def calcular_pontuacoes_test(queries, keys):
    save3 = []

    for i in range(len(queries)):
        save2 = []
        for j in range(len(queries[i])):
            save = []
            for k in range(len(queries[i])):
                save.append(sum(queries[i, j] * keys[i, k]) / 2)

            save2.append(torch.stack(save))

        save3.append(torch.stack(save2))

    return torch.stack(save3)

In [907]:
# New
def calcular_pontuacoes(queries, keys):
    dimensao = keys.shape[-1] ** 0.5
    transposed_keys = keys.transpose(-2, -1)
    return (queries @ transposed_keys) / dimensao

In [908]:
# deprecated
def sliding_mascara(tensors):
    mascaras = []
    for index in range(len(tensors)):
        print(tensors[index])
        mascara = tensors[index].clone()
        mascara[index + 1 : len(mascara)] = float("-inf")
        mascaras.append(mascara)

    return torch.stack(mascaras)

In [909]:
# new
def aplicar_mascara_causal(pontuacoes):
    pontuacoes_mascara = pontuacoes.clone()
    for exemplos_index in range(len(pontuacoes_mascara)):
        tokens_tamanho = len(pontuacoes_mascara[exemplos_index])
        for tokens_index in range(tokens_tamanho):
            pontuacoes_mascara[exemplos_index][tokens_index][tokens_index + 1 : tokens_tamanho] = float("-inf")

    return pontuacoes_mascara


In [910]:
def calcular_pesos(pontuacoes_mascaradas):
    return torch.softmax(pontuacoes_mascaradas, dim=-1)

In [911]:
def combinar_values(pesos, values):
    return pesos @ values

In [912]:
def atencao_causal(queries, keys, values):
    tensor_pe = calcular_pontuacoes(queries, keys)
    pontuacoes_mascaradas = aplicar_mascara_causal(tensor_pe)
    pesos = calcular_pesos(pontuacoes_mascaradas)
    contextos = combinar_values(pesos, values)
    return pesos, contextos

In [913]:
pesos, contextos = atencao_causal(queries, keys, values)

# Testes

In [914]:
test2 = pesos @ values
print(test2.shape)
print(torch.allclose(contextos, test2))

torch.Size([2, 3, 4])
True


# Validação se altera o futuro

In [915]:
queries_clone = queries.clone()
keys_clone = keys.clone()
values_clone = values.clone()

queries_clone[0][2] += 10
keys_clone[0][2] += 10
values_clone[0][2] += 10

In [916]:
pesos_clone, contexto_clone = atencao_causal(queries_clone, keys_clone, values_clone)
print(torch.allclose(contexto_clone[0][0], contextos[0][0]))
print(torch.allclose(contexto_clone[0][1], contextos[0][1]))


True
True


# Melhorando atencao

In [917]:
class AtencaoCausal(torch.nn.Module):
    def __init__(self, dimensao_entrada, dimensao_saida=0):
        super().__init__()
        self.dimensao_entrada = dimensao_entrada
        self.dimensao_saida = (
            dimensao_entrada if dimensao_saida <= 0 else dimensao_saida
        )
        self.camada_linear_query = torch.nn.Linear(
            dimensao_entrada, self.dimensao_saida, bias=False
        )
        self.camada_linear_key = torch.nn.Linear(
            dimensao_entrada, self.dimensao_saida, bias=False
        )
        self.camada_linear_value = torch.nn.Linear(
            dimensao_entrada, self.dimensao_saida, bias=False
        )

    def forward(self, x):
        queries = self.camada_linear_query(x)
        keys = self.camada_linear_key(x)
        values = self.camada_linear_value(x)
        return atencao_causal(queries, keys, values)

In [921]:
atencao_cabeça_1 = AtencaoCausal(4, 2)
atencao_cabeça_2 = AtencaoCausal(4, 2)


In [924]:
peso_atencao_1, contexto_atencao_1 = atencao_cabeça_1(input_vectors)
print(peso_atencao_1, contexto_atencao_1)
peso_atencao_2, contexto_atencao_2 = atencao_cabeça_2(input_vectors)
print(peso_atencao_2, contexto_atencao_2)

tensor([[[1.0000, 0.0000, 0.0000],
         [0.2012, 0.7988, 0.0000],
         [0.5825, 0.1462, 0.2713]],

        [[1.0000, 0.0000, 0.0000],
         [0.8168, 0.1832, 0.0000],
         [0.4416, 0.0848, 0.4736]]], grad_fn=<SoftmaxBackward0>) tensor([[[ 0.8227,  1.0043],
         [ 0.6525,  0.2872],
         [ 1.0928,  0.9884]],

        [[ 0.1741, -0.0698],
         [ 0.4883,  0.2894],
         [ 0.4052,  0.6107]]], grad_fn=<UnsafeViewBackward0>)
tensor([[[1.0000, 0.0000, 0.0000],
         [0.3310, 0.6690, 0.0000],
         [0.2554, 0.3425, 0.4021]],

        [[1.0000, 0.0000, 0.0000],
         [0.3141, 0.6859, 0.0000],
         [0.4461, 0.4770, 0.0769]]], grad_fn=<SoftmaxBackward0>) tensor([[[ 0.1964, -0.9794],
         [-0.6256,  0.3303],
         [-0.7049, -0.2505]],

        [[ 1.0209, -0.1863],
         [-0.0798, -0.8498],
         [ 0.0536, -0.6885]]], grad_fn=<UnsafeViewBackward0>)


In [926]:
peso_atencao_cabeca = torch.concatenate(peso_atencao_1, peso_atencao_2, dim=1)
print(peso_atencao_cabeca)

TypeError: concatenate() received an invalid combination of arguments - got (Tensor, Tensor, dim=int), but expected one of:
 * (tuple of Tensors tensors, int dim = 0, *, Tensor out = None)
 * (tuple of Tensors tensors, name dim, *, Tensor out = None)
